# Welcome — first, what is this thing?

If you've never opened a notebook before, two minutes of orientation:

**What you're looking at is a *Jupyter notebook*, opened in *Google Colab*.** A notebook is a document that mixes blocks of writing (like this one) with blocks of code. The code blocks have a little ▶ button — clicking it runs that code on Google's computer and shows the result right below the block. Think of it as a lab notebook for software: code, results, and notes interleaved on one page.

**Google Colab** is Google's free service for running notebooks in the browser. You don't install anything. You don't need to know Python. Google hands every attendee a temporary virtual computer — and for today's session, a free graphics card (GPU) — to run the demo on. When you close the tab, it all vanishes. Nothing to clean up.

**What you're about to do:** click `Runtime → Run all` once. The notebook will download two Mark Twain novels (*Tom Sawyer* and *Huckleberry Finn*), build a small AI model from scratch, train it on the text, and let it write some of its own Twain-style prose back at you. The whole thing takes about 10 minutes.

**You do not need to read or understand any of the code.** You're welcome to — every block is commented — but the point of the demo is to *watch* the model learn. Numbers will scroll. Random-looking gibberish will appear and slowly become readable English. That's the whole show.

If something looks broken or you get stuck, look up — Mark will get you unstuck.


# Train a Transformer From Scratch in 5 Minutes
**Demystifying AI — Industry Event, May 2026**

You're about to train a ~10-million-parameter character-level transformer on the complete works of Shakespeare. On Colab's free T4 GPU this takes about 4-5 minutes.

## Before you click Run All

1. **Set the runtime to GPU.** `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.
2. **Click `Runtime` → `Run all`.**
3. Scroll to the training cell. Watch the loss fall and the generated samples evolve from random noise into recognizable Shakespearean prose. **That's the demo.**

The entire transformer is in one cell — you can read it. The procedure you'll watch is the same procedure that built GPT-4: ~200,000× smaller, ~10⁷× less data, same procedure.

## What's in this notebook (a quick map)

- **GPU check** — Quick test; fails loudly if no T4 attached
- **Data** — Download Shakespeare, build a character-level vocabulary
- **Hyperparameters** — All knobs in one place
- **The model** — *The whole transformer in one cell.* Four classes, ~150 lines.
- **Training loop** — The cell to watch. Loss falls. Samples evolve.
- **Extended sample** — One more long generation at the end
- **Recap** — Bridge to the next slides


In [ ]:
# -------------------------------------------------------------------
# GPU check
# Transformers are matrix-multiplication machines. CPUs do them one at
# a time; GPUs do thousands in parallel. Training without a GPU would
# take an hour instead of five minutes — so we abort loudly if we
# didn't get one.
# -------------------------------------------------------------------

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, go to Runtime → Change runtime type → "
        "select 'T4 GPU' → Save → then Runtime → Run all again."
    )

device = "cuda"
torch.set_float32_matmul_precision("high")  # small free speedup for matmuls on T4
print(f"GPU ready: {torch.cuda.get_device_name(0)}")
print(f"PyTorch:   {torch.__version__}")


In [ ]:
# -------------------------------------------------------------------
# Data
# We need text to train on. We use two Mark Twain novels —
# Tom Sawyer (1876) and Huckleberry Finn (1884) — both fully public
# domain, distinctive American voice, dialog-heavy.
# Combined they're ~1.3 MB of plain text.
#
# We tokenize at the CHARACTER level. That means our "vocabulary" is
# just the unique characters in the file — letters, punctuation,
# whitespace, etc. Around 80-90 distinct symbols total.
#
# Why character-level instead of word-level or BPE (what real LLMs
# use)? Two reasons:
#   1. The model is tiny enough that a small vocab keeps things
#      manageable — no tokenizer dependency, no vocab management.
#   2. You get to watch the model literally learn that "th" is a
#      common pair and "e" often follows "h" — visible character
#      statistics before the model figures out words.
# -------------------------------------------------------------------

import os
import requests

# === Pluggable corpus configuration ===
# To swap in a different corpus later, just change this list. Each entry
# is (label, Project-Gutenberg-plain-text URL). The fetcher will strip
# Gutenberg header/footer boilerplate and concatenate.
CORPUS_URLS = [
    ("Tom Sawyer",      "https://www.gutenberg.org/cache/epub/74/pg74.txt"),
    ("Huckleberry Finn","https://www.gutenberg.org/cache/epub/76/pg76.txt"),
]
CORPUS_CACHE = "corpus.txt"


def fetch_and_strip_gutenberg(url):
    """Download a Project Gutenberg plain-text file and strip its
    standard header/footer boilerplate so only the actual book text
    remains."""
    raw = requests.get(url, timeout=30).text
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker   = "*** END OF THE PROJECT GUTENBERG EBOOK"
    start = raw.index(start_marker)
    start = raw.index("\n", start) + 1   # move past the START marker line
    end = raw.index(end_marker)
    return raw[start:end].strip()


# Download once; reuse on subsequent runs in the same Colab session.
if not os.path.exists(CORPUS_CACHE):
    parts = []
    for label, url in CORPUS_URLS:
        print(f"Downloading {label}...")
        parts.append(fetch_and_strip_gutenberg(url))
    text = "\n\n".join(parts)
    with open(CORPUS_CACHE, "w", encoding="utf-8") as f:
        f.write(text)
else:
    text = open(CORPUS_CACHE, encoding="utf-8").read()

print(f"\nCorpus length: {len(text):,} characters")
print(f"Sources: {', '.join(label for label, _ in CORPUS_URLS)}")
print(f"\nFirst 300 chars:\n{text[:300]}")

# -------------------------------------------------------------------
# Build the character-level vocabulary.
# stoi = "string to int" — maps each character to an integer ID
# itos = "int to string" — the reverse, for decoding model output
# -------------------------------------------------------------------
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# encode turns a string into a list of integer token IDs
# decode turns a list of integer token IDs back into a string
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

# IMPORTANT — "vocabulary size" is the ALPHABET, not the dataset.
# It's the count of distinct symbols the model has to choose from at
# every position. Shakespeare uses 65 distinct characters total
# (A-Z, a-z, digits, punctuation, space, newline).
# Compare: GPT-4 uses ~100,000 sub-word tokens in its vocabulary —
# same idea, just chunks bigger than single characters.
print(f"\nVocabulary size: {vocab_size} distinct characters (the alphabet)")
print(f"Sample of the alphabet: {''.join(chars[:30])!r} ...")

# -------------------------------------------------------------------
# Train / validation split — 90% / 10%
#
# Why split the data at all? If we train the model on ALL the text
# and then measure how well it does on that same text, we have no way
# to tell whether it actually LEARNED the patterns of Shakespearean
# English — or whether it just MEMORIZED the exact words we showed it.
#
# So we hold 10% of the corpus back, call it the "validation" set
# (abbreviated "val"), and never let the model see it during training.
# Periodically we measure the model's prediction error ("loss") on
# BOTH sets:
#
#   - train loss = how wrong the model is on the text it studied
#   - val   loss = how wrong the model is on text it has NEVER seen
#
# In a healthy training run, both numbers fall together — that means
# the model is learning real patterns that generalize. If train loss
# keeps dropping but val loss starts CLIMBING, that's "overfitting":
# the model is memorizing the training set instead of learning. Like
# a student who memorized last year's exam answers but can't handle
# this year's questions.
#
# (Same train / validate / test idea from the slide earlier in the talk.)
# -------------------------------------------------------------------

# encode the whole corpus into a single long tensor of integer IDs
data = torch.tensor(encode(text), dtype=torch.long)

# slice off the first 90% for training, last 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# These two numbers are the actual dataset: every position the model
# will see during training (or evaluate against). Each one is a single
# character drawn from the 65-symbol alphabet above.
print(f"\nTraining set:            {len(train_data):,} character tokens (model sees these)")
print(f"Held-out validation set: {len(val_data):,} character tokens (model NEVER sees these during training)")


In [ ]:
# -------------------------------------------------------------------
# Hyperparameters
# Every knob on the model and the trainer, in one place. All chosen
# to make the demo finish in ~4-5 min on Colab's free T4 GPU while
# still producing recognizable Shakespearean output.
# -------------------------------------------------------------------

# === MODEL ARCHITECTURE — totals ~10.7M parameters ===
BLOCK_SIZE = 128     # context length: how many tokens the model sees at once
                     # (real LLMs are 100K+; here 128 chars ≈ a few Shakespeare lines)
                     # ATTENTION COST IS QUADRATIC IN THIS — biggest speed lever
N_LAYER    = 6       # number of stacked transformer blocks (depth)
N_HEAD     = 6       # number of attention heads per block (parallel views)
N_EMBD     = 384     # embedding dimension — the "width" of each token's vector
DROPOUT    = 0.2     # randomly zero 20% of activations during training;
                     # forces the model not to over-rely on any one path (regularization)

# === TRAINING ===
BATCH_SIZE     = 64     # how many sequences we process in parallel per step
LEARNING_RATE  = 3e-4   # how big a step we take downhill on each update
                        # (Karpathy's nanoGPT uses 1e-3 with a cosine schedule;
                        #  we keep it fixed for simplicity)
MAX_ITERS      = 3000   # total training steps
EVAL_INTERVAL  = 300    # how often to pause, measure loss, and print a sample
                        # → 11 checkpoints across the run = good demo cadence
EVAL_ITERS     = 20     # how many batches to average over when measuring val loss
SAMPLE_TOKENS  = 200    # how many characters to generate at each eval

# Seed the RNG so the run is reproducible (same random init, same batches).
# Change this and you'll get different (but qualitatively similar) output.
torch.manual_seed(1337)


## The whole transformer, in one cell

Below is a complete, working GPT — about 150 lines of PyTorch, four classes:

1. **`CausalSelfAttention`** — the heart of the transformer. Each token decides how much to "pay attention" to every previous token.
2. **`MLP`** — a small two-layer feed-forward network that runs at each token position.
3. **`Block`** — one transformer layer: LayerNorm → Attention → LayerNorm → MLP, with residual connections.
4. **`GPT`** — the full model: token embeddings + position embeddings → N blocks → final LayerNorm → projection to vocabulary.

The same architecture, scaled up ~200,000×, is what's inside GPT-4. Scroll through and look — the math is not magic.


In [ ]:
# -------------------------------------------------------------------
# The transformer
# Four classes, one model. ~150 lines. The same building blocks every
# modern LLM uses.
# -------------------------------------------------------------------

import torch.nn as nn
from torch.nn import functional as F


# ===================================================================
# 1. CAUSAL SELF-ATTENTION
# Each token looks at every PRIOR token (never the future) and decides
# how much to "pay attention" to each one. This is THE transformer
# innovation — from the 2017 "Attention Is All You Need" paper.
#
# Things to point out as you scroll:
#   - Q, K, V: query, key, value. Three different views of the input.
#     Query = "what am I looking for?"
#     Key   = "what do I represent?"
#     Value = "what info do I carry to whoever attends to me?"
#   - The causal mask: position t can attend to positions 0..t but
#     never to t+1, t+2, ... — no peeking at the future.
#   - Multi-head: we do this several times in parallel with different
#     projections. Different heads end up learning different patterns
#     (one might track subject-verb agreement, another might track
#     quotation marks — emergent, not designed).
# ===================================================================
class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a causal mask."""

    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0  # embedding dim must split evenly across heads

        # One big linear projection that produces Q, K, V all at once.
        # Output is 3× the embedding size; we split it into three pieces.
        self.c_attn = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)

        # After attention, one more projection to mix the heads' outputs back together.
        self.c_proj = nn.Linear(N_EMBD, N_EMBD, bias=False)

        self.dropout = nn.Dropout(DROPOUT)
        self.n_head = N_HEAD
        self.n_embd = N_EMBD

        # The causal mask: a lower-triangular matrix of 1s.
        # Position t can attend to positions 0..t (the 1s) and not to
        # positions t+1.. (the 0s). We'll use this to set "future" scores
        # to -infinity before the softmax.
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE),
        )

    def forward(self, x):
        # x is shape (B, T, C):
        #   B = batch size, T = sequence length, C = embedding dim
        B, T, C = x.shape

        # Project to Q, K, V in one matmul, then split.
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape each from (B, T, C) into (B, n_head, T, head_dim) so
        # each head can attend independently.
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # Attention scores = Q · Kᵀ, scaled by sqrt(head_dim) for stability.
        # Result shape: (B, n_head, T, T) — every token's affinity for every other.
        att = (q @ k.transpose(-2, -1)) / (k.size(-1) ** 0.5)

        # Apply the causal mask: set future positions to -inf so softmax → 0.
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))

        # Softmax along the last dim → attention weights that sum to 1 per row.
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        # Weighted sum of value vectors using the attention weights.
        # Each output position is a blend of all the value vectors it attended to.
        y = att @ v

        # Re-stitch the heads back into one big vector per token, then project.
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


# ===================================================================
# 2. MLP (FEED-FORWARD)
# After attention has mixed information between tokens, each token
# runs through this small per-position network independently. Standard
# pattern: expand 4×, nonlinearity (GELU), project back. This is where
# the model "thinks" about what it attended to.
# ===================================================================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(N_EMBD, 4 * N_EMBD)      # expand
        self.c_proj = nn.Linear(4 * N_EMBD, N_EMBD)    # project back
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        # GELU is a smoother cousin of ReLU; standard in modern transformers.
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


# ===================================================================
# 3. BLOCK
# One transformer layer. Two sub-layers (attention, MLP), each wrapped
# in LayerNorm + residual connection.
#
# The "x = x + ..." pattern is the RESIDUAL CONNECTION — it's why deep
# networks like this are trainable at all. Gradients flow straight
# through the additions even when the sub-layers learn slowly.
# ===================================================================
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(N_EMBD)
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # norm → attend → add back
        x = x + self.mlp(self.ln_2(x))    # norm → MLP    → add back
        return x


# ===================================================================
# 4. THE GPT MODEL
# Tokens come in as integer IDs. We:
#   1. Look up each token's embedding vector
#   2. Add a position embedding (so the model knows where each token sits)
#   3. Run through N_LAYER stacked Blocks
#   4. Final LayerNorm
#   5. Project from embedding space back to vocabulary scores ("logits")
#
# The output `logits` is shape (B, T, vocab_size) — for every position
# in every sequence, a score for every possible next character.
# ===================================================================
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, N_EMBD)   # token → vector
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)   # position → vector
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size, bias=False)  # vector → vocab logits

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Position indices: [0, 1, 2, ..., T-1] for each sequence in the batch.
        pos = torch.arange(T, device=idx.device)

        # Combine token meaning + position. "What you are" + "where you are".
        x = self.tok_emb(idx) + self.pos_emb(pos)

        # Run through every transformer block in sequence.
        x = self.blocks(x)

        # Final normalization, then project to vocabulary scores.
        x = self.ln_f(x)
        logits = self.lm_head(x)

        # If targets provided, compute the loss (used during training).
        # Cross-entropy = "how surprised was the model by the right answer?"
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    # ---------------------------------------------------------------
    # generate — autoregressive sampling.
    # Take the model's prediction for the last position, draw one
    # token from that distribution, append it to the context, repeat.
    # This is what "inference" (a.k.a. running an LLM) actually looks
    # like — token by token, sequential, slow.
    # ---------------------------------------------------------------
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Crop the context to the most recent BLOCK_SIZE tokens
            # (the model can't see beyond its context window).
            idx_cond = idx[:, -BLOCK_SIZE:]

            # Forward pass → get logits, only keep the LAST position's prediction.
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            # Convert logits → probabilities → draw one token.
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

            # Append to the running sequence and continue.
            idx = torch.cat((idx, next_id), dim=1)
        return idx


# Instantiate the model and move it to the GPU.
model = GPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")


## Training — the loop where the model actually learns

This is the cell to watch. Every 300 iterations it pauses, prints the current loss, and generates 200 characters so you can see how the model is doing.

What "training" actually means: pick a random batch of text, ask the model to predict the next character at every position, measure how wrong it was (cross-entropy loss), compute gradients, nudge every parameter a tiny bit in the direction that lowers the loss. Repeat 3000 times.

You will see the loss fall from ~4.2 (effectively random) to ~1.5 (recognizable Twain-style prose). The samples will evolve from `qXz!Bj` to passages like `Tom warn't going to say nothing about the cave to nobody, and Huck he just looked at me and said he reckoned it was the same`.


In [ ]:
# -------------------------------------------------------------------
# Training loop (the main event)
# -------------------------------------------------------------------

import time


# -------------------------------------------------------------------
# get_batch — randomly slice BATCH_SIZE sequences of BLOCK_SIZE+1
# consecutive characters from the dataset. Targets are the same
# sequence shifted by one (the model has to predict the NEXT char
# at every position).
# -------------------------------------------------------------------
def get_batch(split):
    src = train_data if split == "train" else val_data
    idx = torch.randint(len(src) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([src[i : i + BLOCK_SIZE] for i in idx])         # inputs
    y = torch.stack([src[i + 1 : i + 1 + BLOCK_SIZE] for i in idx]) # next-char targets
    return x.to(device), y.to(device)


# -------------------------------------------------------------------
# estimate_loss — pause training, average loss over EVAL_ITERS
# batches each on train and val. Lets us see if val loss tracks
# train loss (good — generalizing) or diverges (bad — overfitting).
# -------------------------------------------------------------------
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            # autocast = let the matmuls run in fp16 instead of fp32 — ~2x faster on T4
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


# -------------------------------------------------------------------
# sample — generate n_tokens new characters starting from `prompt`.
# Used at every eval checkpoint to print a sample of the model's
# current writing ability.
# -------------------------------------------------------------------
def sample(prompt="\n", n_tokens=SAMPLE_TOKENS):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=n_tokens)[0].tolist()
    model.train()
    return decode(out)


# -------------------------------------------------------------------
# Optimizer = the algorithm that decides how to adjust parameters
# based on gradients. AdamW is the modern default for transformers.
# -------------------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# GradScaler is the partner to mixed-precision autocast. Gradients
# computed in fp16 can underflow (become zero) for small values; the
# scaler multiplies the loss by a big number to keep gradients in
# range, then divides back out at the step.
scaler = torch.amp.GradScaler("cuda")

start = time.time()

print("=" * 60)
print("STARTING TRAINING — watch the loss fall and the samples evolve")
print("=" * 60)

# -------------------------------------------------------------------
# THE TRAINING LOOP
# Every iteration:
#   1. Grab a random batch
#   2. Forward pass: compute loss (how wrong are we?)
#   3. Backward pass: compute gradients (which direction lowers loss?)
#   4. Step: nudge parameters in that direction
# Every EVAL_INTERVAL iterations, pause to measure and print.
# -------------------------------------------------------------------
for it in range(MAX_ITERS + 1):

    # --- Pause every EVAL_INTERVAL iters to evaluate and print a sample ---
    if it % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        elapsed = time.time() - start
        print(
            f"\n--- iter {it:5d} | train {losses['train']:.3f} "
            f"| val {losses['val']:.3f} | {elapsed:5.1f}s elapsed ---"
        )
        print(sample(n_tokens=200))
        print()

    # --- One training step ---
    xb, yb = get_batch("train")

    # Forward pass in mixed precision for speed
    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        _, loss = model(xb, yb)

    # Zero out previous gradients (PyTorch accumulates by default)
    optimizer.zero_grad(set_to_none=True)

    # Backward pass — compute gradients of loss w.r.t. every parameter
    scaler.scale(loss).backward()

    # Parameter update — take one step downhill on the loss surface
    scaler.step(optimizer)
    scaler.update()

total = time.time() - start
print(f"\nDONE — trained {MAX_ITERS} iters in {total:.1f} seconds ({total/60:.1f} min).")


## One more long sample

Now that the model is trained, let it write for 1000 characters instead of 200. This gives the audience a longer artifact to admire (or laugh at — semantic coherence is not the goal here, statistical Shakespeare-ness is).


In [ ]:
# -------------------------------------------------------------------
# Extended sample
# Generate 1000 characters of Shakespeare-style text from the trained model.
# -------------------------------------------------------------------
print("=" * 60)
print("EXTENDED SAMPLE — what your model writes after training")
print("=" * 60)
print()
print(sample(prompt="\n", n_tokens=1000))


## What just happened

You trained a ~10-million-parameter transformer from random weights to producing Twain-style prose, in about 10 minutes, on free cloud hardware.

The procedure — gradient descent against a next-token prediction loss — is the **same procedure** that built GPT-4. Yours is ~200,000× smaller and trained on ~10⁷× less data. The recipe scales.

**What's still missing** to make your model Claude or ChatGPT:

1. **Scale** — parameters and data
2. **SFT** — supervised fine-tuning on assistant-style conversations
3. **RLHF** — a reward model trained on human preferences, then used as a critic to nudge the main model toward higher-scoring outputs
4. **Inference-time scaling** — letting the model think longer before answering (the 2024 recipe shift)

Those are the next slides.
